In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/content/hotel_bookings.csv")
df.shape
df.head(20)


ModuleNotFoundError: No module named 'pandas'

In [ ]:
df.shape
print(df.columns.tolist())
print(df.dtypes)

In [ ]:
print(df["hotel"].value_counts())
print(df["customer_type"].value_counts())

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print(missing.head(20))

In [ ]:
print(df["children"].isna().sum())
print(df["country"].isna().sum())

In [ ]:
print(df["agent"].isna().sum())
print(df["company"].isna().sum())

In [ ]:
duplicate_count = df.duplicated().sum()
print("DuplicateRows",duplicate_count)

In [ ]:
print("Negative ADR Rows",(df["adr"]<0).sum())

In [ ]:
print("ZeroNightBookings",(df["stays_in_weekend_nights"]+ df["stays_in_week_nights"]==0).sum())

In [ ]:
analysis_df = df.copy()
analysis_df["children"] = analysis_df["children"].fillna(0)
analysis_df["country"] = analysis_df["country"].fillna("Unknown")


print(analysis_df[["children","country"]].isna().sum())

In [ ]:
analysis_df = analysis_df.drop(columns=["agent","company"],errors="ignore")


In [ ]:
analysis_df["total_nights"] = (
    analysis_df["stays_in_weekend_nights"]+
    analysis_df["stays_in_week_nights"]
)

analysis_df["total_guests"]= (
    analysis_df["adults"]+
    analysis_df["children"]+
    analysis_df["babies"]

)

analysis_df["is_alone"] = (analysis_df["total_guests"]==1).astype(int)

In [ ]:
analysis_df["lead_time_band"] = pd.cut(
    analysis_df["lead_time"],
    bins=[-1,7,30,90,180,np.inf],
    labels = ["0-7 days","8-30days","31-90days","91-180days","181+dys"]
)


analysis_df["stay_length_band"] = pd.cut(
    analysis_df["total_nights"],
    bins = [-1,2,5,10,np.inf],
    labels = ["0-2nights","3-5nights","6-10nights","11+nights"]
)

In [ ]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]
analysis_df["arrival_date_month"] = pd.Categorical(
    analysis_df["arrival_date_month"],
    categories=month_order,
    ordered=True
)


In [ ]:
total_bookings  = len(analysis_df)
canceled_bookings =  analysis_df["is_canceled"].sum()

not_canceled = total_bookings- canceled_bookings

In [ ]:
cancellation_rate = canceled_bookings/total_bookings*100


print(total_bookings)
print(canceled_bookings)
print(not_canceled)
print(round(cancellation_rate,2))

In [ ]:
status_counts =  analysis_df["is_canceled"].map({0:"Not Canceled",1:"Cancelled"}).value_counts

status_counts.plot(kind = "bar",figsize=(7,4))
plt.title("bookingCancellationStatus")
plt.ylabel("NumberofBookings")
plt.xlabel("Status")
plt.show()

In [ ]:
hotel_summary = (
    analysis_df.groupby("hotel")
    .agg(
        bookings = ("is_canceled","size"),
        canceled = ("is_canceled","sum")
    )
)

In [ ]:
hotel_summary["cancelation_rate_pct"]= (
    hotel_summary["canceled"]/hotel_summary["bookings"]/100
).round(2)


print(hotel_summary)

In [ ]:
customer_type = (
    analysis_df.groupby("customer_type")
    .agg(
        bookings=("is_canceled","size"),
        cancellation_rate= ("is_canceled","mean")

    )
)


customer_type["cancellation_rate_pct"] = (
    customer_type["cancellation_rate"]*100
).round(2)

print(customer_type.sort_values("cancellation_rate_pct",ascending=False))

In [ ]:
segment_table = pd.crosstab(
    analysis_df["market_segment"],
    analysis_df["is_canceled"],
    normalize= "index"
)*100

segment_table = segment_table.rename(columns= {0:"not cancelled",1:"cancelled"})
print(segment_table.round(2))

In [ ]:
lead_table = (
    analysis_df.groupby("lead_time_band",observed=False)
    .agg(
        bookings=("is_canceled","size"),
        avg_lead_time = ("lead_time","mean"),
        cancellation_rate = ("is_canceled","mean")
    )
)

lead_table["cancellation_rate_pct"] = (
    lead_table["cancellation_rate_pct"]*100
).round(2)

In [ ]:
staytable = (
    analysis_df.groupby ("stay_lenth_band",observed = False)
    .agg(
        bookings= ("is_canceled","size"),
        average_nights = ("total_nights","mean"),
        cancellation_rate = ("is_canceled","mean")
    )
)



In [ ]:
staytable["cancellation_rate_pact"] = (staytable["cancellation_rate"]*100).round(2)

In [ ]:
partytable = (
    analysis_df.groupby("total_guests")
    .agg(
        bookings = ("is_canceled","size"),
        cancellation_rate = ("is_canceled","mean")
    )
    .query("bookings>=100")


)



In [ ]:
partytable["cancellation_rate_pct"]= (partytable["cancellation_rate"]*100).round(2)

In [ ]:
summary_adr = analysis_df['adr'].describe

print("Max Adr",analysis_df['adr'].max())
print("Meadian Adr",analysis_df['adr'].median())

analysis_df["booking_value_proxy"]= (
    analysis_df["adr"]*analysis_df["total_nights"]
)

In [ ]:
valuebyhotel = (
    analysis_df.groupby("hotel")
    .agg(
        bookings=("booking_value_proxy","size"),
        avg_daily_rate = ("adr","mean"),
        avg_booking_value = ("booking_value_proxy","mean")
    ).round(2)
)